# 04 | The Full Transformer Block
## Sprint 4 — Day 2

**Previous notebook:** `03` — multi-head attention implemented and tested.
We now have all the pieces. This notebook assembles them.

---

A single Transformer encoder block consists of:

1. **Multi-Head Attention** (built in `03`)
2. **Add & Norm** — a residual connection followed by Layer Normalisation
3. **Feed-Forward Network** — two linear layers with a ReLU in between
4. **Add & Norm** — again

We will also cover *why* each of these exists. Residual connections are not
optional — they are what makes deep Transformers trainable. Layer Norm is not
the same as Batch Norm, and the difference matters.

---

**By the end of this notebook you will have:**
- Implemented Layer Normalisation from scratch
- Built the Feed-Forward sublayer
- Assembled a complete, reusable `TransformerBlock` module
- Verified that output shape matches input shape end-to-end

---

*Next notebook: `05_tiny_gpt.ipynb`*

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

d_model     = 64
num_heads   = 8
d_ff        = 256
max_seq_len = 50
vocab_size  = 256
batch_size  = 2
seq_len     = 10
dropout_rate = 0.1

assert d_model % num_heads == 0

def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)
    position = torch.arange(0, max_seq_len).unsqueeze(1).float()
    div_term = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
    PE[:, 0::2] = torch.sin(position / div_term)
    PE[:, 1::2] = torch.cos(position / div_term)
    return PE

class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pe = positional_encoding(max_seq_len, d_model)
    def forward(self, x):
        return self.embedding(x) + self.pe[:x.shape[1], :]

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        attn_out, weights = scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_O(attn_out), weights

embed_layer = InputEmbedding(vocab_size, d_model, max_seq_len)

print("Setup complete.")
print(f"d_model={d_model} | num_heads={num_heads} | d_ff={d_ff}")

Setup complete.
d_model=64 | num_heads=8 | d_ff=256


**LAYER NORMALISATION**

In [2]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

**FEED-FORWARD NETWORK**

In [3]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

**FULL TRANSFORMER BLOCK**

In [4]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ff        = FeedForward(d_model, d_ff)
        self.norm1     = LayerNorm(d_model)
        self.norm2     = LayerNorm(d_model)

    def forward(self, x):
        attn_out, weights = self.attention(x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x, weights

**SANITY CHECK**

In [5]:
dummy_tokens = torch.randint(0, vocab_size, (batch_size, seq_len))
x = embed_layer(dummy_tokens)

block = TransformerBlock(d_model, num_heads, d_ff)
output, weights = block(x)

print(f"Input shape:   {x.shape}")
print(f"Output shape:  {output.shape}")
print(f"Weight shape:  {weights.shape}")

Input shape:   torch.Size([2, 10, 64])
Output shape:  torch.Size([2, 10, 64])
Weight shape:  torch.Size([2, 8, 10, 10])


**VERIFYING RESIDUAL CONNECTION**

In [6]:
with torch.no_grad():
    x_in = embed_layer(dummy_tokens)
    attn_out, _ = block.attention(x_in)
    x_after_residual = x_in + attn_out
    x_after_norm = block.norm1(x_after_residual)

print(f"Input mean:            {x_in.mean().item():.4f}")
print(f"After attention mean:  {attn_out.mean().item():.4f}")
print(f"After residual mean:   {x_after_residual.mean().item():.4f}")
print(f"After layer norm mean: {x_after_norm.mean().item():.4f}")
print(f"After layer norm std:  {x_after_norm.std().item():.4f}")

Input mean:            0.4567
After attention mean:  -0.0069
After residual mean:   0.4498
After layer norm mean: -0.0000
After layer norm std:  0.9925


# Sprint 4 Summary — The Full Transformer Block

---

## What This Sprint Built

A complete, stackable Transformer encoder block assembled from four components:
Multi-Head Attention, residual connections, Layer Normalisation, and a Feed-Forward Network.
This is the fundamental repeating unit of every Transformer architecture.

---

## The Block Architecture

```
Input x
    │
    ├─────────────────────────────┐
    ▼                             │  residual
MultiHeadAttention                │
    │                             │
    └──────────► Add ◄────────────┘
                  │
              LayerNorm
                  │
    ┌─────────────┤
    │             │
    ▼             │  residual
FeedForward       │
    │             │
    └──► Add ◄────┘
          │
      LayerNorm
          │
        Output (same shape as Input)
```

Two sub-layers, each followed by the same **Add & Norm** pattern:

$$x' = \text{LayerNorm}(x + \text{MultiHeadAttention}(x))$$

$$\text{output} = \text{LayerNorm}(x' + \text{FeedForward}(x'))$$

---

## Component 1 — Residual Connections

A residual connection adds the sub-layer's input directly to its output before normalisation.

**The gradient problem it solves:**
In a deep network, gradients travel backward through every layer via backpropagation.
Each layer multiplies the gradient by its local gradient — if these are consistently less than 1,
the gradient shrinks exponentially. Early layers receive near-zero gradients and stop learning.

**How the residual fixes it:**
The derivative of `layer(x) + x` with respect to `x` is `layer_gradient + 1`.
The `+1` term creates a direct gradient highway — signal flows backward through the addition
regardless of what happens inside the layer. Early layers always receive meaningful gradients.

**The identity initialisation benefit:**
At initialisation, layer weights are small so `layer(x) ≈ 0`, meaning `layer(x) + x ≈ x`.
The block starts as an approximate identity — passing input through unchanged — then gradually
learns useful transformations. A far better starting point than a random transformation.

---

## Component 2 — Layer Normalisation

Applied after every residual addition to stabilise the activation distribution.

For each token vector $x \in \mathbb{R}^{d_{model}}$:

$$\mu = \frac{1}{d_{model}} \sum_{i=1}^{d_{model}} x_i \qquad \sigma^2 = \frac{1}{d_{model}} \sum_{i=1}^{d_{model}} (x_i - \mu)^2$$

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \qquad \text{LayerNorm}(x)_i = \gamma\, \hat{x}_i + \beta$$

| Symbol | Role |
|---|---|
| $\mu$ | Mean across $d_{model}$ features for this token |
| $\sigma^2$ | Variance across $d_{model}$ features for this token |
| $\epsilon$ | Small constant ($10^{-6}$) for numerical stability — prevents division by zero |
| $\hat{x}$ | Normalised values — zero mean, unit variance |
| $\gamma$ | Learned scale — initialised to ones |
| $\beta$ | Learned shift — initialised to zeros |

**Why Layer Norm, not Batch Norm:**
Batch Norm normalises across the batch dimension — unreliable for variable-length sequences
and small batch sizes. Layer Norm normalises across the feature dimension *within each token*,
independent of batch size or sequence length. Every token is normalised on its own terms.

**Connecting to RAG:**
In your RAG system you normalised vectors to unit length before computing cosine similarity —
bringing all vectors to the same scale so dot products measured angle, not magnitude.
Layer Norm does the same spirit of thing: it brings activations to a stable, predictable range
so every subsequent layer always receives well-scaled inputs regardless of training depth.

**What the verification cell confirmed:**
- After Layer Norm: mean $\approx 0.0000$ — mean subtraction forces this by definition
- After Layer Norm: std $\approx 1.0000$ — variance division forces this by definition
- After training: $\gamma$ deviates from ones as the model learns optimal per-dimension scaling

---

## Component 3 — Feed-Forward Network

Applied after the first Add & Norm, independently to every token:

$$\text{FFN}(x) = \max(0,\ xW_1 + b_1)\,W_2 + b_2$$

| Parameter | Shape | Role |
|---|---|---|
| $W_1$ | $(d_{model},\ d_{ff})$ | Expand to inner dimension |
| $W_2$ | $(d_{ff},\ d_{model})$ | Compress back to $d_{model}$ |
| $d_{ff}$ | $= 4 \times d_{model}$ | Inner dimension — 4× expansion factor |
| $\text{ReLU}$ | — | Non-linear activation: $\max(0, x)$ |

**Why the FFN exists:**
Attention is a weighted average of Value vectors — a purely linear operation.
No matter how many attention heads you use, the result is a linear combination.
The FFN introduces **non-linearity** via ReLU, giving the block the expressive power
to represent complex, non-linear relationships that pure matrix multiplication cannot.

**What the two roles are:**
- Attention = **communication** — tokens gather context from each other
- FFN = **computation** — each token independently processes its enriched representation

**Why the 4× expansion:**
The larger intermediate dimension $d_{ff}$ gives the network more representational capacity.
Research suggests this layer acts as a key-value memory — storing factual associations
in its weights that the model retrieves during inference.

---

## The Two-Sub-Layer Pattern

| Sub-layer | What it does | After it |
|---|---|---|
| Multi-Head Attention | Tokens communicate — gather global context | Add & Norm |
| Feed-Forward Network | Tokens compute — process gathered context | Add & Norm |

Both sub-layers change the activation distribution significantly.
Both require normalisation to stabilise the next layer's inputs.
Both benefit from residual connections for gradient flow.
The pattern is identical — `output = LayerNorm(input + sublayer(input))` — applied twice.

---

## The Stackable Property

$$\text{Input shape} = (B,\ T,\ d_{model}) = \text{Output shape}$$

This is the critical design property. Output shape equals input shape, so blocks chain freely:

```python
x = block_1(x)   # (B, T, d_model) → (B, T, d_model)
x = block_2(x)   # (B, T, d_model) → (B, T, d_model)
...
x = block_N(x)   # (B, T, d_model) → (B, T, d_model)
```

Residual connections guarantee gradient flow at any depth.
GPT-2 small stacks 12 of these blocks. GPT-3 stacks 96.

---

## Full Component Responsibility Map

| Component | Single responsibility |
|---|---|
| `InputEmbedding` | Token indices → position-aware vectors |
| `MultiHeadAttention` | Every token gathers context from every other token |
| Residual connection | Gradient highway through arbitrary stacking depth |
| `LayerNorm` | Stable activation distribution after each sub-layer |
| `FeedForward` | Non-linear transformation of each token's enriched vector |
| `TransformerBlock` | One stackable, shape-preserving unit combining all of the above |

---

## Key Takeaways

- The Transformer block has two sub-layers: Attention (communicate) and FFN (compute)
- Every sub-layer is wrapped in **Add & Norm** — residual addition followed by Layer Normalisation
- Residual connections solve the vanishing gradient problem and provide identity initialisation
- Layer Norm stabilises activation distributions per-token, independent of batch size
- The FFN introduces the non-linearity the model needs to represent complex patterns
- Input and output shapes are identical — blocks are stackable to arbitrary depth
- All four components are necessary — removing any one breaks either learning or expressivity

---

*Next: Sprint 5 — Building a Tiny GPT.*
*We now stack $N$ Transformer blocks, add causal masking so the model cannot*
*see future tokens, and attach a language model head that maps hidden states*
*to probability distributions over the vocabulary.*